In [ ]:
#%pip install pandas numpy seaborn matplotlib openpyxl

In [ ]:
#%pip install openpyxl

In [9]:
import pandas as pd
import openpyxl

In [70]:
df_creditos_saude = pd.read_excel("Base_Case_Credito_Profissionais_Saude.xlsx")

## **Visão Geral da tabela**
### conhecendo as colunas e valores

In [71]:
df_creditos_saude.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Profissao           2000 non-null   str  
 1   Tempo_Atuacao_Anos  2000 non-null   str  
 2   Faixa_Renda_Mensal  2000 non-null   str  
 3   Tipo_Produto        2000 non-null   str  
 4   Valor_Contrato      2000 non-null   int64
 5   Inadimplente        2000 non-null   bool 
 6   Inadimplente_FLAG   2000 non-null   int64
dtypes: bool(1), int64(2), str(4)
memory usage: 95.8 KB


In [72]:
for coluna in df_creditos_saude.columns:
    if coluna == "Valor_Contrato":
        break
    else:
        print(coluna)
        print(df_creditos_saude[coluna].unique())
        print()

Profissao
<StringArray>
['Dentista', 'Outros', 'Fisioterapeuta', 'Enfermeiro', 'Medico']
Length: 5, dtype: str

Tempo_Atuacao_Anos
<StringArray>
['3-5', '0-2', '10+', '6-10']
Length: 4, dtype: str

Faixa_Renda_Mensal
<StringArray>
['5k-10k', '10k-20k', 'Até 5k', '20k+']
Length: 4, dtype: str

Tipo_Produto
<StringArray>
['Equipamentos', 'Credito Pessoal', 'Capital de Giro']
Length: 3, dtype: str



In [73]:

df_creditos_saude["Faixa_Renda_Mensal"] = df_creditos_saude["Faixa_Renda_Mensal"].replace("5k-10k", "até 10k")
df_creditos_saude["Faixa_Renda_Mensal"] = df_creditos_saude["Faixa_Renda_Mensal"].replace("10k-20k", "até 20k")


In [74]:
for coluna in df_creditos_saude.columns:
    if coluna == "Valor_Contrato":
        break
    else:
        print(coluna)
        print(df_creditos_saude[coluna].unique())
        print()

Profissao
<StringArray>
['Dentista', 'Outros', 'Fisioterapeuta', 'Enfermeiro', 'Medico']
Length: 5, dtype: str

Tempo_Atuacao_Anos
<StringArray>
['3-5', '0-2', '10+', '6-10']
Length: 4, dtype: str

Faixa_Renda_Mensal
<StringArray>
['até 10k', 'até 20k', 'Até 5k', '20k+']
Length: 4, dtype: str

Tipo_Produto
<StringArray>
['Equipamentos', 'Credito Pessoal', 'Capital de Giro']
Length: 3, dtype: str



In [75]:
df_creditos_saude.describe()

,Valor_Contrato,Inadimplente_FLAG
count,2000.000000,2000.000000
mean,49936.894500,0.092000
std,20691.668546,0.289098
min,5000.000000,0.000000
25%,35454.000000,0.000000
50%,49415.500000,0.000000
75%,64341.000000,0.000000
max,120581.000000,1.000000


In [76]:
df_creditos_saude["Valor_Contrato"].min()

np.int64(5000)

In [77]:
df_creditos_saude["Valor_Contrato"].max()

np.int64(120581)

In [78]:
mediana_valor_contrato = df_creditos_saude["Valor_Contrato"].median()
print(f"Mediana do valor de contrato: {mediana_valor_contrato}")

Mediana do valor de contrato: 49415.5


In [79]:
df_creditos_mediana = df_creditos_saude[(df_creditos_saude["Valor_Contrato"] >= 30000) & (df_creditos_saude["Valor_Contrato"] <= 60000)]
df_creditos_mediana = df_creditos_mediana.reset_index(drop=True)

### Entre registros filtrados por valor de contrato próximos da mediana (60k - 30k), apenas 90 deles correspondem a inadimplência, ou seja 8.62% de 1044 registros

In [116]:
df_inadimplencia_mediana = df_creditos_mediana[df_creditos_mediana["Inadimplente"] == True]
df_inadimplencia_mediana = df_inadimplencia_mediana.reset_index(drop=True)
print("Quantidade de Profissionais Inadimplentes:", len(df_inadimplencia_mediana))


Quantidade de Profissionais Inadimplentes: 90


---

In [117]:
total_registros = len(df_inadimplencia_mediana)

### Para os 90 valores presentes em inadimplência conforme contratos próximos a mediana:
- É possível concluir que mais da metade corresponde a valores solicitados para **Crédito Pessoal**
- Enquanto **Capital de Giro** corresponde a menos de 1/5 

| Crédito Pessoal | Equipamentos | Capital de Giro |
|:---------------:|:------------:|:---------------:|
| 46              | 30           | 13              |
| 55,22%          | 33,33%       | 14,44%          |

In [93]:
lista_tipos = ["Credito Pessoal", "Equipamentos", "Capital de Giro"]
for tipo_produto in lista_tipos:
    contador_tipo = 0
    for linha_contador in range(len(df_inadimplencia_mediana)):
        registro_contador = df_inadimplencia_mediana.loc[linha_contador, "Tipo_Produto"]
        if tipo_produto == registro_contador:
            contador_tipo+=1
    print(tipo_produto)
    print("Quantidade de registros:", contador_tipo)
    print(f"Valor em porcentagem: {round((contador_tipo * 100) / total_registros, 2)}%")
    print()
    

Credito Pessoal
Quantidade de registros: 47
Valor em porcentagem: 52.22%

Equipamentos
Quantidade de registros: 30
Valor em porcentagem: 33.33%

Capital de Giro
Quantidade de registros: 13
Valor em porcentagem: 14.44%



---

### Com base nessa análise, é possível afirmar que:
- a quantidade de inadimplência é bem distribuída entre as profissões, revelando uma diferença mínima entre eles. 
- Apesar disso, Médicos e Dentista tendenciam a ser maioria dentro de inadimplentes.

| Médico          | Dentista     | Fisiote         | Enfermeiro      |  Outros         |
|:---------------:|:------------:|:---------------:|:---------------:|:---------------:|
| 22              | 20           | 17              |   17            |  14             | 
| 24,4%           | 22,22%       | 18,89%          |   19,89%        |  15,56%         | 


In [95]:
profissoes = ['Dentista', 'Outros', 'Fisioterapeuta', 'Enfermeiro', 'Medico']
for profissao in profissoes:
    contador_prof = 0
    for indice_linha in range(len(df_inadimplencia_mediana)):
        registro = df_inadimplencia_mediana.loc[indice_linha, "Profissao"]
        if profissao == registro:
            contador_prof+=1
    print(profissao)
    print("Quantidade de registros:", contador_prof)
    print(f"Valor em porcentagem: {round((contador_prof * 100)/total_registros,2)}%")
    print()

Dentista
Quantidade de registros: 20
Valor em porcentagem: 22.22%

Outros
Quantidade de registros: 14
Valor em porcentagem: 15.56%

Fisioterapeuta
Quantidade de registros: 17
Valor em porcentagem: 18.89%

Enfermeiro
Quantidade de registros: 17
Valor em porcentagem: 18.89%

Medico
Quantidade de registros: 22
Valor em porcentagem: 24.44%



---

### Fica comprovado que profissionais com **renda mensal** menor que 10k, representam maior número de inadimplentes. 
- É evidênciado que a renda mensal tem impacto direto na inadimplência do profissional.

| Até 5k          | 5k-10k       | 10k-20k         | 20k+            | até 10k |
|:---------------:|:------------:|:---------------:|:---------------:|:---------------:|
| 29              | 32           | 18              |   11            | 61 |
| 32,22%          | 35,56%       | 20%             |   12,22%        |67,78%|



In [120]:
renda_mensal = ['Até 5k', 'até 10k', 'até 20k', '20k+']
for renda in renda_mensal:
    contador_renda = 0
    for indice_linha in range(len(df_inadimplencia_mediana)):
        registro = df_inadimplencia_mediana.loc[indice_linha, "Faixa_Renda_Mensal"]
        if renda == registro:
            contador_renda+=1
    print(renda)
    print("Quantidade de registros:", contador_renda)
    print(f"Valor em porcentagem: {round((contador_renda * 100)/ total_registros,2)}%")
    print()

Até 5k
Quantidade de registros: 29
Valor em porcentagem: 32.22%

até 10k
Quantidade de registros: 32
Valor em porcentagem: 35.56%

até 20k
Quantidade de registros: 18
Valor em porcentagem: 20.0%

20k+
Quantidade de registros: 11
Valor em porcentagem: 12.22%



---

### **Aqui fica comprovado que em 80% das profissões:**
- a maioria da inadimplência vem da solicitação do **Crédito Pessoal**, exceto em profissionais de Fisioterapia

In [113]:
linhas = []

for profissao in profissoes:
    df_temp_profissao = df_inadimplencia_mediana[df_inadimplencia_mediana["Profissao"] == profissao].copy()
    df_temp_profissao = df_temp_profissao.reset_index(drop=True)
    total_registros = len(df_temp_profissao)

    for tipo_contrato in lista_tipos:
        contador_tipo = 0
        for indice_linha in range(len(df_temp_profissao)):
            registro_tipo = df_temp_profissao.loc[indice_linha, "Tipo_Produto"]
            if tipo_contrato == registro_tipo:
                contador_tipo += 1

        linhas.append({
            "Profissao": profissao,
            "Tipo_Contrato": tipo_contrato,
            "Quant. Registros": contador_tipo,
            "%": round((contador_tipo * 100) / total_registros, 2),
        })

df_profissionais = pd.DataFrame(linhas)

for profissao in profissoes:
    print(df_profissionais[df_profissionais["Profissao"] == profissao])
    print()

  Profissao    Tipo_Contrato  Quant. Registros     %
0  Dentista  Credito Pessoal                 9  45.0
1  Dentista     Equipamentos                 7  35.0
2  Dentista  Capital de Giro                 4  20.0

  Profissao    Tipo_Contrato  Quant. Registros      %
3    Outros  Credito Pessoal                 8  57.14
4    Outros     Equipamentos                 3  21.43
5    Outros  Capital de Giro                 3  21.43

        Profissao    Tipo_Contrato  Quant. Registros      %
6  Fisioterapeuta  Credito Pessoal                 7  41.18
7  Fisioterapeuta     Equipamentos                 8  47.06
8  Fisioterapeuta  Capital de Giro                 2  11.76

     Profissao    Tipo_Contrato  Quant. Registros      %
9   Enfermeiro  Credito Pessoal                11  64.71
10  Enfermeiro     Equipamentos                 3  17.65
11  Enfermeiro  Capital de Giro                 3  17.65

   Profissao    Tipo_Contrato  Quant. Registros      %
12    Medico  Credito Pessoal                

---